In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
# NOTE: change this path to the folder where you saved the downloaded workbook
raw = Path.home() / "Desktop/Tableau/DataSources/online_retail_ii"
sheets = pd.read_excel(raw / "online_retail.xlsx", sheet_name=None)
print(list(sheets))


In [ ]:
#02 load the datam count nulls
for name, df in sheets.items():
    print(name, df.shape)
    print(df.dtypes)
    print("dates:", df["InvoiceDate"].min(), "->", df["InvoiceDate"].max())
    print(df.isna().sum(), "\n")

In [ ]:
#03 check sheet columns if they are the same
a, b = sheets.values()
print(list(a.columns) == list(b.columns))

In [ ]:
# check the null valued Customer IDs 
a, b = sheets.values()
df = pd.concat([a, b], ignore_index=True)
df["revenue"] = df["Quantity"] * df["Price"]
no_id = df["Customer ID"].isna()

print("share of rows:", no_id.mean())
print(df.groupby(no_id)["revenue"].sum())
print("price == 0:", (df.loc[no_id, "Price"] == 0).sum())
print("quantity < 0:", (df.loc[no_id, "Quantity"] < 0).sum())

In [ ]:
g = df.groupby("Invoice")["Customer ID"]
share_null = g.apply(lambda s: s.isna().mean())
partial = share_null[(share_null > 0) & (share_null < 1)]
print("invoices:", len(share_null))
print("partly null:", len(partial))

In [6]:
# Cell 4: Save raw CSVs
for name, df_s in sheets.items():
    df_s.to_csv(raw / f"{name.replace(' ', '_')}.csv", index=False)

In [ ]:
# Cell 5: Check invoice overlap
a, b = sheets.values()
common = set(a["Invoice"]) & set(b["Invoice"])
print("shared invoices:", len(common))
print("rows in sheet 1:", a["Invoice"].isin(common).sum())
print("rows in sheet 2:", b["Invoice"].isin(common).sum())

In [60]:
# Cell 7: Partly-null invoices check
g = df.groupby("Invoice")["Customer ID"]
share_null = g.apply(lambda s: s.isna().mean())
partial = share_null[(share_null > 0) & (share_null < 1)]
print("invoices:", len(share_null))
print("partly null:", len(partial))

invoices: 53628
partly null: 0


In [ ]:
# Cell 8: Verify overlap identical
cols = list(a.columns)
sa = a[a["Invoice"].isin(common)].sort_values(cols).reset_index(drop=True)
sb = b[b["Invoice"].isin(common)].sort_values(cols).reset_index(drop=True)
print(sa.equals(sb))

In [62]:
# Cell 9: Combine, drop overlap
b_unique=b[~b['Invoice'].isin(common)]
df_all=pd.concat([a,b_unique],ignore_index=True)
len(a), len(b_unique), len(df_all)

(525461, 519387, 1044848)

In [63]:
#10 find negative Quantity
neg_quan=df_all[df_all['Quantity'] <0]
neg_quan.tail(10)

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
1043079,C581468,21314,SMALL GLASS HEART TRINKET POT,-10,2011-12-08 19:26:00,2.10,13599.0,United Kingdom
1043080,C581468,22098,BOUDOIR SQUARE TISSUE BOX,-12,2011-12-08 19:26:00,0.39,13599.0,United Kingdom
1043081,C581470,23084,RABBIT NIGHT LIGHT,-4,2011-12-08 19:28:00,2.08,17924.0,United Kingdom
1043360,C581484,23843,"PAPER CRAFT , LITTLE BIRDIE",-80995,2011-12-09 09:27:00,2.08,16446.0,United Kingdom
1043386,C581490,22178,VICTORIAN GLASS HANGING T-LIGHT,-12,2011-12-09 09:57:00,1.95,14397.0,United Kingdom
1043387,C581490,23144,ZINC T-LIGHT HOLDER STARS SMALL,-11,2011-12-09 09:57:00,0.83,14397.0,United Kingdom
1044479,C581499,M,Manual,-1,2011-12-09 10:28:00,224.69,15498.0,United Kingdom
1044653,C581568,21258,VICTORIAN SEWING BOX LARGE,-5,2011-12-09 11:57:00,10.95,15311.0,United Kingdom
1044654,C581569,84978,HANGING HEART JAR T-LIGHT HOLDER,-1,2011-12-09 11:58:00,1.25,17315.0,United Kingdom
1044655,C581569,20979,36 PENCILS TUBE RED RETROSPOT,-5,2011-12-09 11:58:00,1.25,17315.0,United Kingdom


In [ ]:
# 11 check one specific example
test=df_all[df_all['Invoice']=='C489459']
print (test.sort_values('Description'))

In [ ]:
# 12 check one specific example
test=df_all[df_all['Invoice']=='C581568']
print (test.sort_values('Description'))

In [ ]:
#13 neg quantity invoices
neg_quan["Invoice"].value_counts(),len(neg_quan)

In [65]:
#14 invoices start with C and check them with negative quantities
is_c = df_all["Invoice"].astype(str).str.startswith("C")
is_neg = df_all["Quantity"] < 0
pd.crosstab(is_c, is_neg)

Quantity,False,True
Invoice,,
False,1022290,3393
True,1,19164


In [ ]:
#15 Invoice counts starts with C
is_c.sum()

np.int64(19165)

In [66]:
# 17 Find negative not C starting -is_neg is True and is_c is False
print(neg_quan['Price'].describe())
print( (neg_quan['Price']==0).sum())

count    22557.000000
mean        36.285245
std        529.206074
min          0.000000
25%          0.850000
50%          2.250000
75%          4.950000
max      38970.000000
Name: Price, dtype: float64
3393


In [67]:
#18: Zero price in group
grp=df_all[~is_c&is_neg]
print(len(grp))
print((grp['Price']==0).mean())

3393
1.0


In [68]:
#19: Group nulls and descriptions
print(grp['Customer ID'].isna().mean())
print(grp['Description'].value_counts().head(10))

1.0
Description
check                     121
damages                    83
?                          81
damaged                    78
missing                    27
sold as set on dotcom      20
Damaged                    17
smashed                     9
thrown away                 9
Unsaleable, destroyed.      9
Name: count, dtype: int64


In [69]:
#20 check the invoices is repeated
c_inv = set(df_all.loc[is_c, "Invoice"].astype(str).str[1:])
n_inv = set(df_all.loc[~is_c, "Invoice"].astype(str))
print(len(c_inv), len(c_inv & n_inv))

8292 0


In [ ]:
#21 Create keys and check if they are duplicated
print(df_all.duplicated(subset=['Invoice','StockCode']).sum())
print(df_all.duplicated(subset=['Invoice', 'StockCode', 'Customer ID']).sum())

In [ ]:
#22 One Customer per invoice
df_all.groupby('Invoice')['Customer ID'].nunique().max()

In [ ]:
#23 Exact duplicates
df_all.duplicated().sum()

In [ ]:
#24 Look at examples
rep = df_all[df_all.duplicated(subset=["Invoice", "StockCode"], keep=False)]
rep.sort_values(["Invoice", "StockCode"]).head(10)

In [72]:
# Cell 24: Revenue of exact duplicates
dups = df_all[df_all.duplicated()]
rev_dups = (dups["Quantity"] * dups["Price"]).sum()
rev_all = (df_all["Quantity"] * df_all["Price"]).sum()
print(rev_dups, rev_all, rev_dups / rev_all)

54228.420000000006 18909762.117999997 0.0028677473392634887


In [73]:
# 25 Check o price with C invoice
conditions = [is_c, is_neg]
labels     = ["cancellation", "adjustment"]
df_all["row_type"] = np.select(conditions, labels, default="sale")
print(df_all['row_type'].value_counts())

row_type
sale            1022290
cancellation      19165
adjustment         3393
Name: count, dtype: int64


In [28]:
#26 Exact duplicates
columns=['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate',
       'Price', 'Customer ID', 'Country']
df_all["is_exact_duplicate"]=df_all.duplicated(subset=columns)
print(df_all["is_exact_duplicate"].sum())

11812


In [29]:
#27 Sort and give unique ID to each row
df_all=df_all.sort_values(['InvoiceDate','Invoice']).reset_index(drop=True)
df_all['RowID']=range(1, len(df_all) +1)
print(df_all['RowID'].is_unique, df_all['RowID'].max())

True 1044848


In [ ]:
#28 count Price 0
zp=df_all[(df_all['row_type']=='sale')& (df_all['Price']==0 )]
print(len(zp))
print(zp["Customer ID"].isna().mean())
print(zp["Description"].value_counts().head(10))

In [ ]:
#29 Label zero-price rows
df_all.loc[(df_all["row_type"] == "sale") & (df_all["Price"] == 0), "row_type"] = "zero_price"
print(df_all["row_type"].value_counts())

In [ ]:
# 30 Revenue by row_type
df_all["Revenue"] = df_all["Quantity"] * df_all["Price"]
df_all.groupby("row_type")["Revenue"].sum()

In [ ]:
# Cell 31: Non-product codes
code = df_all["StockCode"].astype(str)
nonprod = df_all[~code.str.match(r"^\d")]
print(len(nonprod))
print(nonprod.groupby("StockCode")["Revenue"].agg(["count", "sum"]).sort_values("count", ascending=False).head(15))

In [ ]:
#32 Sample description per code
top = nonprod["StockCode"].value_counts().head(20).index
nonprod[nonprod["StockCode"].isin(top)].drop_duplicates("StockCode")[["StockCode", "Description"]]

In [74]:
# 33: All non-digit codes
print(sorted(nonprod["StockCode"].unique()))

['ADJUST', 'ADJUST2', 'AMAZONFEE', 'B', 'BANK CHARGES', 'C2', 'C3', 'CRUK', 'D', 'DCGS0003', 'DCGS0004', 'DCGS0006', 'DCGS0016', 'DCGS0027', 'DCGS0036', 'DCGS0037', 'DCGS0039', 'DCGS0041', 'DCGS0044', 'DCGS0053', 'DCGS0055', 'DCGS0056', 'DCGS0057', 'DCGS0058', 'DCGS0059', 'DCGS0060', 'DCGS0062', 'DCGS0066N', 'DCGS0066P', 'DCGS0067', 'DCGS0068', 'DCGS0069', 'DCGS0070', 'DCGS0071', 'DCGS0072', 'DCGS0073', 'DCGS0074', 'DCGS0075', 'DCGS0076', 'DCGSLBOY', 'DCGSLGIRL', 'DCGSSBOY', 'DCGSSGIRL', 'DOT', 'GIFT', 'M', 'PADS', 'POST', 'S', 'SP1002', 'TEST001', 'TEST002', 'gift_0001_10', 'gift_0001_20', 'gift_0001_30', 'gift_0001_40', 'gift_0001_50', 'gift_0001_60', 'gift_0001_70', 'gift_0001_80', 'gift_0001_90', 'm']


In [ ]:
# 34: Check unclear codes
unclear = ["ADJUST2", "B", "C3", "GIFT", "SP1002", "TEST002", "m",
           "DCGSLBOY", "DCGSLGIRL", "DCGSSBOY", "DCGSSGIRL"]
chk = df_all[df_all["StockCode"].astype(str).isin(unclear)]
chk.drop_duplicates(["StockCode", "Description"])[["StockCode", "Description"]].sort_values("StockCode")

In [ ]:
#  35: Inspect C3 and GIFT
df_all[df_all["StockCode"].isin(["C3", "GIFT"])][["StockCode", "Quantity", "Price", "Customer ID", "InvoiceDate", "row_type"]]

In [76]:
#36 Build item_type.

shipping   = ["POST", "DOT", "C2"]
accounting = ["D", "M", "m", "ADJUST", "ADJUST2", "B", "S", "BANK CHARGES", "AMAZONFEE", "CRUK"]
test       = ["TEST001", "TEST002"]
unknown    = ["C3", "GIFT"]

code = df_all["StockCode"].astype(str)
conditions = [code.isin(shipping), code.isin(accounting), code.isin(test),
              code.isin(unknown), code.str.startswith("gift_0001_")]
labels = ["shipping", "accounting", "test", "unknown", "gift_voucher"]
df_all["item_type"] = np.select(conditions, labels, default="product")
print(df_all['item_type'].value_counts())


item_type
product         1039035
shipping           3788
accounting         1906
gift_voucher        100
test                 17
unknown               2
Name: count, dtype: int64


In [ ]:
# 37: Fix data types.
df_all['StockCode']=df_all['StockCode'].astype(str)
df_all['Customer ID']=df_all['Customer ID'].astype('Int64')
print(df_all.dtypes)

In [ ]:
#38: Invoice to text
df_all["Invoice"].map(type).value_counts()

In [ ]:
# Cell 38b: Odd text invoices
odd = df_all[df_all["Invoice"].map(type).eq(str) & ~df_all["Invoice"].astype(str).str.startswith("C")]
odd[["Invoice", "StockCode", "Description", "Quantity", "Price", "item_type"]]

In [77]:
#38c: Impact of A invoices
print(odd["Revenue"].sum())
print(odd["row_type"].value_counts())

-147614.08
row_type
sale    6
Name: count, dtype: int64


In [43]:
#39 Date columns
df_all["InvoiceDay"] = df_all["InvoiceDate"].dt.normalize()
df_all["InvoiceMonth"] = df_all["InvoiceDate"].dt.to_period("M").dt.to_timestamp()

In [78]:
# 40: Row type × item type
pd.crosstab(df_all['item_type'],df_all['row_type'])

row_type,adjustment,cancellation,sale
item_type,,,
accounting,0,948,958
gift_voucher,0,1,99
product,3391,17974,1017670
shipping,0,238,3550
test,0,4,13
unknown,2,0,0


In [ ]:
#  41: Relabel accounting rows
df_all.loc[df_all["item_type"] == "accounting", "row_type"] = "adjustment"
print(pd.crosstab(df_all["item_type"], df_all["row_type"]))

In [ ]:
#  42: Check new columns
df_all["Invoice"] = df_all["Invoice"].astype(str)
print(df_all["Invoice"].map(type).value_counts())
print(df_all[["InvoiceDate", "InvoiceDay", "InvoiceMonth"]].head())
print(df_all[["InvoiceDay", "InvoiceMonth"]].dtypes)

In [ ]:
#  43: Rename date columns
df_all = df_all.rename(columns={"InvoiceDate": "InvoiceDateTime", "InvoiceDay": "InvoiceDate"})
print(df_all.columns.tolist())

In [48]:
#44b Rename and reorder
df_all = df_all.rename(columns={
    "Customer ID": "CustomerID",
    "row_type": "RowType",
    "item_type": "ItemType",
    "is_exact_duplicate": "IsExactDuplicate",
})

order = ["RowID", "Invoice", "InvoiceDateTime", "InvoiceDate", "InvoiceMonth",
         "CustomerID", "Country", "StockCode", "Description",
         "ItemType", "RowType", "IsExactDuplicate",
         "Quantity", "Price", "Revenue"]

df_all = df_all[order]
print(df_all.shape)

(1044848, 15)


In [ ]:
#  45: Save checkpoint CSV
out = Path.home() / "Desktop/Tableau/online-retail-ii/data_clean"
df_all.to_csv(out / "online_retail_checkpoint.csv", index=False)

In [ ]:
#46 Cohort month
cond = (df_all["CustomerID"].notna()) & (df_all["RowType"] == "sale") & (df_all["ItemType"] == "product")
coh = df_all[cond].groupby("CustomerID")["InvoiceMonth"].min()
df_all["CohortMonth"] = df_all["CustomerID"].map(coh)

In [ ]:
#47 check cohort
print(coh.shape)
print(df_all["CohortMonth"].isna().mean())
print(coh.value_counts().sort_index().head())

In [54]:
# 48: Why no cohort?
print(df_all['CustomerID'].isna().mean())

0.22518777850941


In [57]:
#  49: Final clean export
final_dir = Path("/home/emil/Desktop/DataSources/OnlineRetail")
final_dir.mkdir(parents=True, exist_ok=True)
df_all.to_csv(final_dir / "online_retail_ii_clean.csv", index=False)
print(df_all.shape)

(1044848, 16)


In [ ]:
# 50 Months since cohort
cb = df_all[cond].copy()
cb["MonthIndex"] = ((cb["InvoiceMonth"].dt.year - cb["CohortMonth"].dt.year) * 12
                    + (cb["InvoiceMonth"].dt.month - cb["CohortMonth"].dt.month))
print(cb["MonthIndex"].describe())

In [59]:
#  51: Cohort table
cohort = (cb.groupby(["CohortMonth", "MonthIndex"])["CustomerID"]
            .nunique().reset_index(name="Customers"))
cohort["MonthIndex"] = cohort["MonthIndex"].astype(int)

size = cohort[cohort["MonthIndex"] == 0].set_index("CohortMonth")["Customers"]
cohort["CohortSize"] = cohort["CohortMonth"].map(size)
cohort["Retention"] = (cohort["Customers"] / cohort["CohortSize"]).round(4)

cohort.to_csv(final_dir / "online_retail_ii_cohort.csv", index=False)
print(cohort.shape)
print(cohort.head())

(325, 5)
  CohortMonth  MonthIndex  Customers  CohortSize  Retention
0  2009-12-01           0        951         951     1.0000
1  2009-12-01           1        333         951     0.3502
2  2009-12-01           2        317         951     0.3333
3  2009-12-01           3        404         951     0.4248
4  2009-12-01           4        360         951     0.3785
